This notebook is for Core persona extraction:

clustering, archetypes, sentiment, behavioral profiles

In [ ]:
import pandas as pd
import numpy as np

print("Environment working")

In [ ]:
import pandas as pd
import numpy as np

# File paths
review_path = "../data/raw/yelp_academic_dataset_review.json"
business_path = "../data/raw/yelp_academic_dataset_business.json"
user_path = "../data/raw/yelp_academic_dataset_user.json"
print("Paths loaded successfully")

In [ ]:
# STEP 3
# Loading dataframes
# For initial exploration, we will load a sample of the reviews and users datasets to avoid memory issues, while loading the entire business dataset to analyze restaurant-related information.
# This approach allows us to get a sense of the structure and content of the datasets without overwhelming our system's memory, while still providing us with enough data to perform meaningful analysis on restaurant reviews and business information.

reviews_sample = pd.read_json(
    review_path,
    lines=True,
    nrows=5000
)

business_df = pd.read_json(
    business_path,
    lines=True,
)

users_sample = pd.read_json(
    user_path,
    lines=True,
    nrows=5000
)

print("Reviews shape:", reviews_sample.shape)
print("Business dataset shape:")
print(business_df.shape)
print("Users shape:", users_sample.shape)

In [ ]:
# STEP 4
# Inspecting columns of each dataset
# This will help us understand the structure of the data and identify key features for analysis.

print("REVIEWS COLUMNS")
print(reviews_sample.columns)

print("\nBUSINESS COLUMNS")
print(business_df.columns)

print("\nUSER COLUMNS")
print(users_sample.columns)

In [ ]:
# STEP 5
# Displaying first few rows of each dataset to get a sense of the data
reviews_sample.head(2)

In Reviews, there are important fields like: user_id, business_id, stars, text, date

These serves as our behavioral signal layer.

In [ ]:
# STEP 6
# Displaying first few rows of business dataset

business_df.head(2)

In Businesses

Fields like: categories,city, attributes,stars

This serves as our contextual environment layer.

In [ ]:

# STEP 7
# Displaying first few rows of users dataset

users_sample.head(2)

In Users fields like: review_count, average_stars, fans

These serves as our behavioral metadata layer.

STEP 8 
Filtering Restaurant Businesses

We only want businesses related to: restaurants, food, cafes, bars because this domain contains rich emotional/social behavior.

In [ ]:
# Keep only businesses with restaurant-related categories
# This will help us focus our analysis on the restaurant industry, which is a major part of Yelp's business and user interactions.
# filters businesses whose categories contain: Restaurant, Food, Coffee, Cafe, Bar

restaurant_businesses = business_df[
    business_df["categories"]
    .fillna("")
    .str.contains(
        "Restaurant|Food|Coffee|Cafe|Bar",
        case=False,
        regex=True
    )
]

print("Restaurant businesses:")
print(restaurant_businesses.shape)

In [ ]:
# STEP 9
# Inspect Restaurant businesses

restaurant_businesses[
    ["business_id", "name", "categories", "city"]
].head(10)

In [ ]:
# STEP 10
# Extract unique restaurant business IDs

restaurant_ids = set(
    restaurant_businesses["business_id"]
)

print("Number of restaurant IDs:")
print(len(restaurant_ids))

In [ ]:
# STEP 11
# Filter reviews to include only those related to the restaurant businesses identified above. 
# This will allow us to analyze user feedback specifically for restaurants, which is crucial for understanding customer satisfaction and business performance in this sector.
# This makes personas cleaner, preferences clearer, recommendations better

restaurant_reviews = reviews_sample[
    reviews_sample["business_id"].isin(
        restaurant_businesses["business_id"]
    )
]

print("Restaurant reviews shape:")
print(restaurant_reviews.shape)

In [ ]:
# STEP 12
#  Inspect restaurant reviews
# This will give us insights into the types of feedback customers are providing for restaurants, which can inform our analysis of customer satisfaction and business performance in this sector.

restaurant_reviews[
    ["user_id", "business_id", "stars", "text"]
].head(5)

Finally, I am looking at real behavioral data. I can now see:

emotions
complaints
enthusiasm
sarcasm
value judgments

This is a raw psychological signal.

STEP 13: Find Sweet-Spot Users

Now we identify behaviorally rich users.

In [ ]:
# Count reviews per user
# This will help us identify active users and understand the distribution of reviews among users, which can inform our analysis of user behavior and preferences in the restaurant sector.

user_review_counts = (
    restaurant_reviews
    .groupby("user_id")
    .size()
    .reset_index(name="review_count")
)

# Keep users with 15–50 reviews

sweet_spot_users = user_review_counts[
    (user_review_counts["review_count"] >= 15) &
    (user_review_counts["review_count"] <= 50)
]

print("Sweet-spot users:")
print(sweet_spot_users.shape)

In [ ]:
# STEP 13B
# Inspect sweet-spot users
sweet_spot_users.head(10)

In [ ]:
# STEP 14
# Read review data in chunks
# This approach allows us to process large datasets without running into memory issues, enabling us to filter and analyze reviews related to restaurants efficiently.
# By reading the review data in chunks, we can handle the large size of the dataset while still extracting relevant information for our analysis of restaurant reviews.

chunk_size = 100000

restaurant_review_chunks = []

for chunk in pd.read_json(
    review_path,
    lines=True,
    chunksize=chunk_size
):
    
    filtered_chunk = chunk[
        chunk["business_id"].isin(restaurant_ids)
    ]
    
    restaurant_review_chunks.append(filtered_chunk)

    print(
        f"Processed chunk with "
        f"{len(filtered_chunk)} restaurant reviews"
    )

In [ ]:
# Step 15
# Concatenate all filtered chunks into a single DataFrame
# This will give us a complete dataset of restaurant reviews that we can use for further analysis, 
# such as sentiment analysis, user behavior analysis, and business performance evaluation in the restaurant sector.
# This is our real behavioral corpus.

restaurant_reviews = pd.concat(
    restaurant_review_chunks,
    ignore_index=True
)

print("Final restaurant reviews shape:")
print(restaurant_reviews.shape)

In [ ]:
# STEP 16
# Count reviews per user
# This will help us identify users who have reviewed a moderate number of restaurants, which might be indicative of engaged reviewers.

user_review_counts = (
    restaurant_reviews
    .groupby("user_id")
    .size()
    .reset_index(name="review_count")
)

sweet_spot_users = user_review_counts[
    (user_review_counts["review_count"] >= 15) &
    (user_review_counts["review_count"] <= 50)
]

print("Sweet-spot users:")
print(sweet_spot_users.shape)

In [ ]:
# STEP 17
# Curate a sample of sweet-spot users for further analysis, 
# ensuring we have a manageable number of users to work with while still capturing a representative subset of engaged reviewers.

curated_users = sweet_spot_users.sample(
    n=300,
    random_state=42
)

print(curated_users.shape)

In [ ]:
# Step 18
# Filter restaurant reviews to include only those from the curated sweet-spot users.
# This will allow us to focus our analysis on a specific subset of engaged users, which can provide more meaningful insights into user behavior and preferences in the restaurant sector.

curated_user_ids = set(
    curated_users["user_id"]
)

curated_reviews = restaurant_reviews[
    restaurant_reviews["user_id"]
    .isin(curated_user_ids)
]

print("Curated reviews shape:")
print(curated_reviews.shape)

This reveals a curated corpus of: psychologically rich users, restaurant behaviors, emotional language, preferences, review styles

Enough to build: personas, review simulation, recommendations, conversational reasoning

In [ ]:
# STEP 19
# Transforming raw reviews into interpretable human behavioral traits.
# First, we organize and group reviews by user and business, then we can analyze patterns in the review text, star ratings, and other features to derive insights about user preferences, sentiment, and engagement with restaurants. This will help us understand the underlying behaviors and traits of users in the context of restaurant reviews.



user_histories = (
    curated_reviews
    .groupby("user_id")
    .agg({
        "text": list,
        "stars": list,
        "business_id": list,
        "date": list
    })
    .reset_index()
)

print("User histories shape:")
print(user_histories.shape)

In [ ]:
# STEP 20
# Inspecting a sample user history to understand the structure of the data and the type of information we have for each user, which will help us in deriving behavioral traits and insights from their reviews.
# Change the index to inspect different users and their review histories.

sample_user = user_histories.iloc[70]

print("USER ID:")
print(sample_user["user_id"])

print("\nSTAR RATINGS:")
print(sample_user["stars"][:5])

print("\nFIRST REVIEW:")
print(sample_user["text"][0][:500])

By changing the index, i notice behavioral traces of a real human.

Things i notice:

tone
complaints
enthusiasm
emotional intensity
writing style
priorities

This is where personas emerge.

In [ ]:
# STEP 21 — Creating First Persona Features
# Here we are calculating basic features for each user based on their reviews, such as average rating, rating variance, review count, and average review length. 
# These features can help us understand user behavior and preferences in the context of restaurant reviews, which can be useful for building user personas and improving recommendation systems.

persona_features = (
    curated_reviews
    .groupby("user_id")
    .agg(
        avg_rating=("stars", "mean"),
        rating_variance=("stars", "std"),
        review_count=("stars", "count"),
        avg_review_length=(
            "text",
            lambda x: np.mean(
                x.str.len()
            )
        )
    )
    .reset_index()
)

print(persona_features.shape)

persona_features.head()

WHAT THESE FEATURES MEAN
  Feature	               Psychological Meaning
- avg_rating	           harsh vs lenient
- rating_variance	       emotional consistency
- review_count	           engagement
- avg_review_length	       verbosity/detail orientation

These are behavioral signals.

In [ ]:
# STEP 22 — Adding Sentiment Features
# Now we estimate emotional tone.

In [ ]:
from textblob import TextBlob
from tqdm import tqdm

tqdm.pandas()

In [ ]:
# Sentiment per review

curated_reviews["sentiment"] = (
    curated_reviews["text"]
    .progress_apply(
        lambda x: TextBlob(x).sentiment.polarity
    )
)

In [ ]:
# STEP 23
# Aggregating User Sentiment

sentiment_features = (
    curated_reviews
    .groupby("user_id")
    .agg(
        avg_sentiment=("sentiment", "mean"),
        sentiment_variance=("sentiment", "std")
    )
    .reset_index()
)

sentiment_features.head()

WHAT THESE MEAN
Feature	              Meaning
avg_sentiment	      positivity/negativity
sentiment_variance	  emotional stability

In [ ]:
# STEP 24
# Merge Persona Features

persona_df = persona_features.merge(
    sentiment_features,
    on="user_id"
)

print(persona_df.shape)

persona_df.head()

In [ ]:
# STEP 25 — Interpret Personas

persona_df.describe()

This uncovers human archetypes.

Examples:

angry critics
generous reviewers
emotional storytellers
concise pragmatists

In [ ]:
# Harsh Users
# These users tend to give lower ratings on average, which may indicate a more critical perspective or higher standards when it comes to restaurant experiences.

persona_df.sort_values(
    by="avg_rating"
).head(5)

In [ ]:
# Lenient Users
# These users tend to give higher ratings on average, which may indicate a more positive outlook or a tendency to be more forgiving in their reviews.

persona_df.sort_values(
    by="avg_rating",
    ascending=False
).head(5)

In [ ]:
# Verbose Users
# These users tend to write longer reviews, which may indicate a higher level of engagement or a desire to provide more detailed feedback.

persona_df.sort_values(
    by="avg_review_length",
    ascending=False
).head(5)

With these, we can:

characterize users
compare personalities
simulate tendencies
reason about preferences

Next, We will transform numeric persona signals into recognizable human archetypes.

Examples:

harsh critic
emotional foodie
soft-life explorer
concise pragmatist
luxury seeker

This becomes our Dynamic Cognitive Persona Layer.

In [ ]:
# STEP 26 — Prepare for Clustering
# Here we are selecting the relevant features for clustering and filling any missing values with 0. 
# This will allow us to group users into distinct personas based on their review behavior and sentiment, which can be useful for targeted marketing, personalized recommendations, and understanding customer segments in the restaurant industry.

clustering_features = persona_df[
    [
        "avg_rating",
        "rating_variance",
        "avg_review_length",
        "avg_sentiment",
        "sentiment_variance"
    ]
].fillna(0)

clustering_features.head()

In [ ]:
# STEP 27 — Standardize Features / Scale Features for Clustering
# Standardizing features is crucial for clustering algorithms, especially those that rely on distance metrics (like K-Means), as it ensures that all features contribute equally to the distance calculations.

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_features = scaler.fit_transform(
    clustering_features
)

print(scaled_features.shape)

In [ ]:
# STEP 28 — Create Behavioral Clusters with K-Means
# K-Means is a popular clustering algorithm that partitions data into K distinct clusters based on feature similarity. 
# By applying K-Means to our standardized features, we can identify distinct user personas based on their review behavior and sentiment, which can provide valuable insights for targeted marketing and personalized recommendations in the restaurant industry.

from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=5,
    random_state=42
)

persona_df["cluster"] = kmeans.fit_predict(
    scaled_features
)

persona_df.head()

WHY THIS MATTERS

The model is now grouping users by:

emotional style
harshness
verbosity
behavioral consistency

This is latent human behavior discovery.

In [ ]:
# STEP 29 — Analyze Cluster Distribution. 
# This will help us understand how users are grouped into different personas based on their review behavior and sentiment, 
# which can inform our strategies for targeted marketing, personalized recommendations, and customer segmentation in the restaurant industry.

persona_df["cluster"].value_counts()

In [ ]:
# STEP 30 — Summarize Cluster Characteristics
# This will allow us to understand the defining features of each cluster, which can help us interpret the underlying behaviors 
# and traits of users in each persona, providing insights that can inform targeted marketing strategies, personalized recommendations, and customer segmentation in the restaurant industry.
# Understand Each Cluster AND compute average traits per cluster.

cluster_summary = (
    persona_df
    .groupby("cluster")
    [
        [
            "avg_rating", # 
            "avg_review_length", 
            "avg_sentiment",
            "rating_variance"
        ]
    ]
    .mean()
)

cluster_summary

In [ ]:
# STEP 31 - Assign Human Archetype Names
# Interpret Clusters with Names


cluster_names = {
    0: "Warm Optimist",
    1: "Reactive Reviewer",
    2: "Harsh Critic",
    3: "Emotional Storyteller",
    4: "Deep Experience Analyst"
}

persona_df["archetype"] = (
    persona_df["cluster"]
    .map(cluster_names)
)

persona_df.head()

In [ ]:
# STEP 31A - Create Structured Behavioral Descriptors
# 

behavioral_descriptors = {

    0: {
        "archetype": "Warm Optimist",

        "traits": {
            "positivity": "high",
            "verbosity": "moderate",
            "emotional_stability": "stable",
            "review_style": "supportive",
            "expectation_level": "moderate",
            "decision_style": "emotionally positive"
        }
    },

    1: {
        "archetype": "Reactive Reviewer",

        "traits": {
            "positivity": "mixed",
            "verbosity": "moderate",
            "emotional_stability": "volatile",
            "review_style": "emotion-driven",
            "expectation_level": "variable",
            "decision_style": "experience-sensitive"
        }
    },

    2: {
        "archetype": "Harsh Critic",

        "traits": {
            "positivity": "low",
            "verbosity": "high",
            "emotional_stability": "critical",
            "review_style": "analytical",
            "expectation_level": "high",
            "decision_style": "detail-oriented"
        }
    },

    3: {
        "archetype": "Emotional Storyteller",

        "traits": {
            "positivity": "moderate",
            "verbosity": "high",
            "emotional_stability": "reflective",
            "review_style": "narrative",
            "expectation_level": "balanced",
            "decision_style": "emotionally expressive"
        }
    },

    4: {
        "archetype": "Deep Experience Analyst",

        "traits": {
            "positivity": "moderate",
            "verbosity": "very high",
            "emotional_stability": "stable",
            "review_style": "deeply descriptive",
            "expectation_level": "high",
            "decision_style": "deliberative"
        }
    }
}

In [ ]:
# STEP 31B — Attach Archetype Names
persona_df["archetype"] = (
    persona_df["cluster"]
    .apply(
        lambda x: behavioral_descriptors[x]["archetype"]
    )
)

persona_df.head()

In [ ]:
# STEP 31C — Attach Structured Traits
# Add the full trait dictionaries.


persona_df["behavior_profile"] = (
    persona_df["cluster"]
    .apply(
        lambda x: behavioral_descriptors[x]["traits"]
    )
)

persona_df.head()

WHAT THE DATAFRAME NOW CONTAINS

Each user now has:

Column	                Meaning
avg_rating	            rating behavior
avg_review_length	    verbosity
avg_sentiment	        emotional tone
archetype	            human-readable identity
behavior_profile	    machine-readable cognition

In [ ]:
# Qualitative Inspection of User Reviews by Archetype
for archetype in persona_df["archetype"].unique():

    print("\n" + "="*80)
    print(f"ARCHETYPE: {archetype}")
    print("="*80)

    sampled_users = (
        persona_df[
            persona_df["archetype"] == archetype
        ]
        .sample(2, random_state=42)   # fewer users
    )

    for _, user_row in sampled_users.iterrows():

        sample_user_id = user_row["user_id"]

        print("\n" + "#"*60)
        print(f"USER ID: {sample_user_id}")
        print("#"*60)

        user_reviews = curated_reviews[
            curated_reviews["user_id"] == sample_user_id
        ]

        for i, (_, row) in enumerate(
            user_reviews.head(1).iterrows(),  # only 1 review
            start=1
        ):

            print("\n" + "-"*50)
            print(f"Review #{i}")
            print(f"Stars: {row['stars']}")
            print("-"*50)

            print(row["text"][:200].replace("\n", " ") + "...")
            print("\n")

In [ ]:
# STEP 32 — Final Persona Summary
persona_df[
    ["user_id", "archetype"]
].head(10)

In [ ]:
# STEP 33- Merge Archetypes Back Into Reviews
curated_reviews = curated_reviews.merge(
    persona_df[
        ["user_id", "archetype"]
    ],
    on="user_id",
    how="left"
)

curated_reviews.head()

In [ ]:
# STEP 34 — Define Value Taxonomy
# This taxonomy will help us categorize and analyze the different aspects of restaurant reviews, 
# allowing us to extract meaningful insights about customer values, preferences, expectations, and experiences. 
# For each review it counts how often value-related language appears.

import re

def extract_value_signals(text, taxonomy):

    text = text.lower()

    scores = {}

    for category, keywords in taxonomy.items():

        score = 0

        for keyword in keywords:

            matches = re.findall(
                rf"\b{re.escape(keyword)}\b",
                text
            )

            score += len(matches)

        scores[category] = score

    return scores

In [ ]:
# STEP 34A — Define Value Taxonomy
# Here we are defining a taxonomy of value-related keywords that can be used to analyze reviews.

value_taxonomy = {

    # Economic values
    "affordability": [
        "cheap", "affordable", "budget", "expensive", "overpriced", "pricey",
        "value for money", "manage", "cost", "naira", "save cost", "waste of money",
        "not worth it", "fair price", "discount", "original price", "market price"
    ],


    # Durability & quality expectations
    "durability": [
        "original", "fake", "counterfeit", "rugged", "last long", "strong",
        "weak", "fragile", "repair", "spare parts", "generator", "battery life",
        "heat up", "spoilt", "still working", "tested and trusted", "tokunbo", "new", "used"
    ],


    # Service & staff behaviour (restaurants, repairs, delivery)
    "service_quality": [
        "service", "staff", "waiter", "waitress", "attentive", "rude",
        "friendly", "slow service", "fast response", "customer care",
        "come and fix", "collect and vanish", "follow me around", "attention to detail", "attentive"
        "pushy seller", "polite", "helpful", "unhelpful", "knowledgeable", "incompetent", "courteous", "disrespectful"
    ],


    # Social proof & communal influence
    "social_proof": [
        "neighbour", "friend recommended", "landlord use am", "colleague",
        "family said", "word of mouth", "everybody buying", "popular",
        "trending", "see my neighbour", "trusted by many", "influencer", "celebrity endorsement", "social media hype"
    ],


    # Temporal / efficiency norms
    "time_efficiency": [
        "wait time", "delay", "fast delivery", "slow", "African time",
        "hours", "minutes", "late", "early", "prompt", "wasted my time",
        "traffic", "Lagos traffic", "delivered on time", "arrived late", 
        "arrived early", "on schedule", "behind schedule", "ahead of schedule"
    ],

    # Ambience & atmosphere (restaurants, events)
    "ambience": [
        "ambience", "atmosphere", "decor", "vibes", "music", "aesthetic", "cozy"
        "lighting", "cleanliness", "noise level", "comfortable seating", "romantic", 
        "family-friendly", "decoration choke", "overcrowded", "spacious", "intimate", "loud", "quiet"
    ],


    # Product‑specific attributes (food, electronics, fashion, books, etc.)
    "food_quality": [
        "delicious", "bland", "taste", "fresh", "flavor", "authentic", "portion",
        "presentation", "spicy", "sweet", "sour", "fresh", "salty", "umami", "overcooked",
        "undercooked", "stale", "rotten", "mouthwatering", "swallow", "fufu", "eba", 
        "soup", "rice", "portion size", "small",
    ],


    "electronics": [
        "generator", "inverter", "phone", "laptop", "battery", "charger",
        "NEPA", "light", "plug", "heating", "original charger", "waterproof"
        "power bank", "noise cancelling", "wireless", "durable", "fast charging", "long battery life"
    ],


    "fashion": [
        "lace", "ankara", "fabric quality", "zipper", "tailor", "sewn",
        "fit", "size", "colour fast", "shrink", "native wear", "casual wear", 
        "formal wear", "workwear", "party wear", "traditional attire"
    ],


    "books_media": [
        "motivational", "hustle", "inspirational", "educational", "story",
        "grammar", "chapters", "cover", "print quality", "prayer points"
    ],

    # Convenience & accessibility
    "convenience": [
        "fast", "quick", "parking", "location", "accessible", "easy",  "waiting time"
        "near me", "home delivery", "takeaway", "drive-thru", "curbside pickup", "self-service"
    ],

    # Social experience
    "social_experience": [
        "friends", "family", "date", "group", "celebration", "birthday", "hangout"
        "social gathering", "romantic dinner", "family outing", "friend meetup", "special occasion"
        "anniversary", "reunion", "casual hangout", "work event", "holiday celebration"
        "new spot to try", "place to see and be seen", "vibe for socializing", "perfect for groups", "intimate setting"
    ],

    # Aspirational & status‑related cues
    "luxury": [
        "premium", "luxury", "upscale", "fancy", "high-end", "exclusive", "luxurious", "opulent", 
        "lavish", "posh", "sophisticated", "elegant", "glamorous"
    ],

   # Sentiment polarity markers (Nigerian style exaggeration)
    "positive_exaggeration": [
        "the best", "amazing", "perfect", "excellent", "I love am",
        "changed my life", "highly recommend", "must buy" "refreshing", "life-changing", "unforgettable", "top-notch", "five stars", "beyond expectations"
    ],

    "negative_exaggeration": [
        "worst ever", "terrible", "useless", "waste of data", "I spit",
        "never again", "run away", "scam", "fake life"
    ],


    # Cold‑start & exploration signals
    "uncertainty": [
        "not sure", "maybe", "let me test", "first time", "trying",
        "I no know o", "time will tell", "hoping for the best", "heard good things", "heard bad things", "mixed reviews", "on the fence"
    ]

}

In [ ]:
sample_review = curated_reviews.iloc[7]["text"]

print(sample_review[:500])

extract_value_signals(
    sample_review,
    value_taxonomy
)

IMPORTANT

This is interpretable behavioral reasoning. It enables us to explain why the system believes something.

In [ ]:
# STEP 35 - APPLY VALUE SIGNAL EXTRACTION
# Now we apply this function to all reviews to extract value signals for each review, 
# which will allow us to analyze the presence of different value-related themes in the 
# reviews and understand customer preferences.

from tqdm import tqdm

tqdm.pandas()

curated_reviews["value_signals"] = (
    curated_reviews["text"]
    .progress_apply(
        lambda x: extract_value_signals(
            x,
            value_taxonomy
        )
    )
)

In [ ]:
# STEP 36 — EXPAND VALUE SIGNALS INTO COLUMNS
# This will allow us to analyze the presence of different value-related themes in the reviews and 
# understand customer preferences in a more structured way, enabling us to identify patterns and 
# insights related to the values expressed in the reviews.

value_df = pd.json_normalize(
    curated_reviews["value_signals"]
)

value_df.head(12)

In [ ]:
# STEP 37 — MERGE VALUE FEATURES
# Now each review contains inferred human values.
# This will allow us to analyze the presence of different value-related themes in the reviews 
# and understand customer preferences in a more structured way, enabling us to identify patterns 
# and insights related to the values expressed in the reviews.

curated_reviews = pd.concat(
    [curated_reviews, value_df],
    axis=1
)

curated_reviews.head()



In [ ]:
# STEP 38 — AGGREGATE VALUES PER USER
# Now we infer what users fundamentally care about.
# By aggregating the value signals at the user level, we can identify overarching themes and preferences 
# that characterize each user's reviews, providing deeper insights into their values and priorities when it 
# comes to restaurant experiences.

user_values = (
    curated_reviews
    .groupby("user_id")
    [
        list(value_taxonomy.keys())
    ]
    .mean()
    .reset_index()
)

user_values.head()

In [ ]:
# STEP 39 — MERGE VALUES INTO PERSONAS
# This will allow us to enrich our user personas with insights about the values that are most important to them, 
# which can inform targeted marketing strategies, personalized recommendations, and a deeper understanding of customer segments.

persona_df = persona_df.merge(
    user_values,
    on="user_id",
    how="left"
)

persona_df.head()

In [ ]:
# STEP 40 — IDENTIFY DOMINANT VALUES
# This will help us understand the key values that define each user persona, allowing us to tailor our marketing strategies and recommendations to align with what matters most to each segment of our customer base.
# We identify the dominant value for each user by finding the value category with the highest average score in their reviews, which can provide insights into their core preferences and priorities.

value_columns = list(value_taxonomy.keys())

persona_df["dominant_value"] = (
    persona_df[value_columns]
    .idxmax(axis=1)
)

persona_df[
    ["user_id", "archetype", "dominant_value"]
].head(11)

Personas are no longer generic reviewers. They now contain:

- emotional behavior
- archetypes
- behavioral style
- human priorities
- dominant values

PHASE 2 — TEMPORAL & EMOTIONAL DRIFT MODELING

This is where the system learns that humans evolve.

We will model:

Behavior	              Meaning
- rating drift	          becoming harsher/more generous over time
- sentiment drift	      emotional evolution
- preference drift	      changing tastes
- temporal behavior	      seasonality
- identity evolution	  life-stage transitions

In [ ]:
# STEP 41 — PREPARE TEMPORAL DATA
# We first ensure dates are proper datetime objects.

# Convert review dates to datetime

curated_reviews["date"] = pd.to_datetime(
    curated_reviews["date"]
)

print("Date conversion complete")

In [ ]:
# STEP 42 — SORT USER REVIEWS CHRONOLOGICALLY


curated_reviews = curated_reviews.sort_values(
    by=["user_id", "date"]
)

curated_reviews.head()

WHY THIS MATTERS

Human evolution only makes sense over time.

Now reviews become behavioral timelines, instead of isolated events.

In [ ]:
# STEP 43 — CREATE TEMPORAL USER HISTORIES
# Here we are creating temporal user histories by aggregating the review dates, star ratings, and sentiment scores for each user into lists.
# This will allow us to analyze how user preferences and sentiments evolve over time, providing insights into user behavior and trends in reviews.

temporal_histories = (
    curated_reviews
    .groupby("user_id")
    .agg({
        "date": list,
        "stars": list,
        "sentiment": list
    })
    .reset_index()
)

temporal_histories.head()

WHAT THIS REPRESENTS

Each user now has:

chronological ratings
emotional trajectory
behavioral timeline

This represents a temporal identity.

In [ ]:
# STEP 44 — COMPUTE RATING DRIFT
# We now measure whether users become harsher or softer over time.
# By computing the rating drift, we can identify trends in user behavior, 
# such as whether users tend to become more critical or more lenient in their reviews over time, 
# which can provide insights into changes in user expectations and satisfaction.

def compute_rating_drift(ratings):

    if len(ratings) < 2:
        return 0

    return ratings[-1] - ratings[0]

In [ ]:
temporal_histories["rating_drift"] = (
    temporal_histories["stars"]
    .apply(compute_rating_drift)
)

temporal_histories.head()

In [ ]:
# STEP 45 — COMPUTE SENTIMENT DRIFT
# We now measure the evolution of user sentiments over time.
# By computing the sentiment drift, we can identify trends in user emotions, 
# such as whether users tend to become more positive or more negative in their reviews over time, 
# which can provide insights into changes in user experiences and satisfaction.

def compute_sentiment_drift(sentiments):

    if len(sentiments) < 2:
        return 0

    return sentiments[-1] - sentiments[0]

In [ ]:
temporal_histories["sentiment_drift"] = (
    temporal_histories["sentiment"]
    .apply(compute_sentiment_drift)
)

temporal_histories.head()

WHAT THIS MEANS

We are now detecting emotional evolution.

Examples:
- increasing positivity
- frustration accumulation
- declining enthusiasm
- emotional fatigue

In [ ]:
# STEP 46 — CLASSIFY EMOTIONAL TRAJECTORIES
# We classify users based on their sentiment drift to understand their emotional trajectories over time.

def classify_drift(value):

    if value > 0.5:
        return "becoming_more_positive"

    elif value < -0.5:
        return "becoming_more_negative"

    else:
        return "emotionally_stable"

In [ ]:
temporal_histories["emotional_trajectory"] = (
    temporal_histories["sentiment_drift"]
    .apply(classify_drift)
)

temporal_histories.head(10)

In [ ]:
# STEP 47 — SUMMARIZE EMOTIONAL TRAJECTORIES
# This will help us understand the distribution of emotional trajectories among users, 
# providing insights into how user sentiments evolve over time and whether there are common patterns in emotional changes.

stable = (temporal_histories["emotional_trajectory"] == "emotionally_stable").sum()
positive = (temporal_histories["emotional_trajectory"] == "becoming_more_positive").sum()
negative = (temporal_histories["emotional_trajectory"] == "becoming_more_negative").sum()

print("emotionally_stable:", stable)
print("becoming_more_positive:", positive)
print("becoming_more_negative:", negative)

In [ ]:
# STEP 48 — INSPECT USERS WITH EMOTIONAL DRIFT
# This will allow us to qualitatively analyze the reviews of users who exhibit significant emotional drift, 
# providing insights into the factors that may contribute to changes in user sentiment over time and how these changes manifest in their reviews.

for trajectory in [
    "becoming_more_positive",
    "becoming_more_negative"
]:

    print("\n" + "="*80)
    print(f"TRAJECTORY: {trajectory}")
    print("="*80)

    # Get sample users
    sampled_users = (
        temporal_histories[
            temporal_histories["emotional_trajectory"]
            == trajectory
        ]
        .sample(3, random_state=42)
    )

    for _, user_row in sampled_users.iterrows():

        user_id = user_row["user_id"]

        print("\n" + "#"*70)
        print(f"USER ID: {user_id}")
        print(f"Trajectory: {trajectory}")
        print(f"Rating Drift: {user_row['rating_drift']}")
        print(f"Sentiment Drift: {user_row['sentiment_drift']}")
        print("#"*70)

        # Get chronological reviews
        user_reviews = (
            curated_reviews[
                curated_reviews["user_id"] == user_id
            ]
            .sort_values("date")
        )

        # FIRST REVIEW
        first_review = user_reviews.iloc[0]

        print("\nFIRST REVIEW")
        print("-"*50)
        print(f"Date: {first_review['date']}")
        print(f"Stars: {first_review['stars']}")
        print(f"Sentiment: {first_review['sentiment']}")

        print(first_review["text"][:500])

        # LAST REVIEW
        last_review = user_reviews.iloc[-1]

        print("\nLAST REVIEW")
        print("-"*50)
        print(f"Date: {last_review['date']}")
        print(f"Stars: {last_review['stars']}")
        print(f"Sentiment: {last_review['sentiment']}")

        print(last_review["text"][:500])

        print("\n\n")

In [ ]:
# STEP 49 — MERGE TEMPORAL SIGNALS INTO PERSONAS
# This will allow us to enrich our user personas with insights about how their sentiments and ratings evolve over time,

persona_df = persona_df.merge(
    temporal_histories[
        [
            "user_id",
            "rating_drift",
            "sentiment_drift",
            "emotional_trajectory"
        ]
    ],
    on="user_id",
    how="left"
)

persona_df.head()

WHAT WE NOW POSSESS: A real cognitive persona engine.

The personas now include:

Dimension	         Meaning
archetype	         behavioral identity
dominant_value	     priorities
sentiment	         emotionality
drift	             evolution
trajectory	         temporal behavior




PHASE 3: PREFERENCE DRIFT MODELLING

This models changing interests over time.

Example:

2019:
cheap fast food

2023:
upscale aesthetic dining

That implies income change, identity change, maturity, lifestyle evolution etc.

We will detect:

- changing restaurant categories
- changing review topics
- evolving priorities
- shifting preferences

In [ ]:
# STEP 50 — EXTRACT BUSINESS CATEGORIES
# 

business_subset = business_df[
    ["business_id", "categories"]
].copy()

business_subset.head()

In [ ]:
# STEP 51 — MERGE BUSINESS CATEGORIES INTO REVIEWS

curated_reviews = curated_reviews.merge(
    business_subset,
    on="business_id",
    how="left"
)

curated_reviews.head()

In [ ]:
# STEP 52 — CLEAN CATEGORY TEXT

curated_reviews["categories"] = (
    curated_reviews["categories"]
    .fillna("")
)

In [ ]:
# STEP 53 — CREATE SIMPLE CUISINE TAGS
# We now infer preference domains.


cuisine_keywords = [
    "Mexican",
    "Italian",
    "Chinese",
    "Japanese",
    "Thai",
    "Indian",
    "American",
    "Mediterranean",
    "Korean",
    "French",
    "Pizza",
    "Seafood",
    "Burgers",
    "Cafe",
    "Bars"
]

In [ ]:
# STEP 54 — EXTRACT CUISINE PREFERENCES

def extract_cuisines(category_text):

    found = []

    for cuisine in cuisine_keywords:

        if cuisine.lower() in category_text.lower():
            found.append(cuisine)

    return found

In [ ]:
curated_reviews["cuisines"] = (
    curated_reviews["categories"]
    .apply(extract_cuisines)
)

curated_reviews[
    ["categories", "cuisines"]
].head()

In [ ]:
# STEP 55 — SPLIT EARLY VS RECENT BEHAVIOR
# We compare past self vs current self.

def split_temporal_preferences(user_df):

    user_df = user_df.sort_values("date")

    midpoint = len(user_df) // 2

    early = user_df.iloc[:midpoint]
    recent = user_df.iloc[midpoint:]

    return early, recent

In [ ]:
# STEP 56: DEYECT PREFERENCE DRIFT

from collections import Counter

preference_drift_results = []

for user_id, user_df in curated_reviews.groupby("user_id"):

    if len(user_df) < 6:
        continue

    early, recent = split_temporal_preferences(user_df)

    early_cuisines = [
        cuisine
        for sublist in early["cuisines"]
        for cuisine in sublist
    ]

    recent_cuisines = [
        cuisine
        for sublist in recent["cuisines"]
        for cuisine in sublist
    ]

    early_top = Counter(early_cuisines).most_common(3)
    recent_top = Counter(recent_cuisines).most_common(3)

    preference_drift_results.append({
        "user_id": user_id,
        "early_preferences": early_top,
        "recent_preferences": recent_top
    })

In [ ]:
# STEP 57 — CREATE PREFERENCE DRIFT DATAFRAME

preference_drift_df = pd.DataFrame(
    preference_drift_results
)

preference_drift_df.head()

In [ ]:
# Inspect a few users
preference_drift_df.sample(15)

Possible interpretation for preference drift of these users:

maturing tastes, luxury orientation, lifestyle evolution

This shows human evolution modeling.

PHASE 4 — LINGUISTIC STYLE MODELING
This is critical for Task A review simulation. Because humans are not only what they value or how they feel. They are also how they speak.

In this phase, we are building an agent that understands varying styles and dimensions to review writing	
Example:
- storytelling (long narrative reviews)
- analytical	(structured critique)
- emotionality	(expressive language)
- sarcasm	(indirect criticism)
- slang	(casual speech)
- enthusiasm	(exaggerated positivity)
- formality	(polished vs casual)

Build:

- review style metrics
- punctuation behavior
- capitalization style
- emotional intensity
- storytelling tendency
- slang detection
- expressiveness

In [ ]:
# STEP 58 — CREATE REVIEW LENGTH FEATURE
# This will allow us to analyze how the length of reviews correlates with user personas, sentiments, and preferences, providing insights into user engagement and expression in their reviews.

curated_reviews["review_length"] = (
    curated_reviews["text"]
    .apply(len)
)

In [ ]:
# STEP 59 — EXCLAMATION USAGE
# This captures emotional expressiveness.
# By counting the number of exclamation marks in reviews, we can gain insights into the emotional intensity and 
# expressiveness of users, which may correlate with certain personas or sentiments in their reviews.

curated_reviews["exclamation_count"] = (
    curated_reviews["text"]
    .str.count("!")
)

In [ ]:
# STEP 60 — QUESTION MARK USAGE
# This captures sarcasm, confusion, and conversational tone
# By counting the number of question marks in reviews, we can gain insights into the conversational tone in user reviews, 
# which may correlate with certain personas or sentiments.

curated_reviews["question_count"] = (
    curated_reviews["text"]
    .str.count(r"\?")
)

In [ ]:
# STEP 61 — UPPERCASE EMPHASIS
# Humans use caps for excitemenT, anger, emphasis
# By calculating the ratio of uppercase letters in reviews, we can gain insights into the emotional intensity and 
# emphasis in user reviews, which may correlate with certain personas or sentiments.

def uppercase_ratio(text):

    if len(text) == 0:
        return 0

    uppercase_chars = sum(
        1 for c in text if c.isupper()
    )

    return uppercase_chars / len(text)

In [ ]:
curated_reviews["uppercase_ratio"] = (
    curated_reviews["text"]
    .apply(uppercase_ratio)
)

In [ ]:
# STEP 62 — STORYTELLING DETECTION

storytelling_keywords = {
    
    # Temporal sequencing (story moves through time)
    "time_sequence": [
        "first", "then", "next", "after that", "finally", "later",
        "before", "when", "while", "as soon as", "suddenly",
        "eventually", "in the end", "at first", "initially"
    ],
    
    # Personal experience framing
    "personal_anchor": [
        "I arrived", "I went", "I walked in", "I met", "I saw",
        "I decided", "I asked", "I told", "I thought", "I felt",
        "my friend and I", "we entered", "they welcomed us"
    ],
    
    # Scene setting (time, place, atmosphere)
    "scene_setting": [
        "it was a", "the weather", "the place was", "the atmosphere",
        "as soon as I entered", "the smell of", "the music was playing",
        "it was crowded", "quiet", "bustling", "dark", "bright"
    ],
    
    # Dialogue / quoted speech (strong storytelling signal)
    "dialogue": [
        "said", "asked", "told me", "shouted", "whispered",
        "called me", "replied", "answered", "he said '", "she said '",
        "I said '", "then he goes '", "I was like '"
    ],
    
    # Emotional arc (build‑up, climax, resolution)
    "emotion_arc": [
        "I was excited", "disappointed", "surprised", "shocked",
        "relieved", "angry", "happy", "sad", "confused",
        "my heart sank", "I couldn't believe", "I almost cried",
        "I laughed", "I regretted", "I was so happy that"
    ],
    
    # Conflict & resolution (classic story structure)
    "conflict_resolution": [
        "problem was", "issue came up", "something went wrong",
        "unfortunately", "luckily", "thankfully", "to make matters worse",
        "in the end", "they fixed it", "we sorted it out", "I complained"
    ],
    
    # Nigerian‑specific storytelling markers (colloquial narrative style)
    "naija_narrative": [
        "so I tell am", "the guy come say", "I just dey go",
        "immediately I enter", "see as e be", "before I know",
        "the next thing", "as I was coming", "I reach there",
        "the seller tell me", "my sister say make I try am",
        "story for another day", "I no fit shout", "you won't believe"
    ],
    
    # Exaggerated storytelling (typical of oral tradition)
    "hyperbole_narrative": [
        "I waited for years", "the longest hour of my life",
        "everybody in Lagos", "the whole market", "I almost died",
        "I swear down", "e be like film", "like a movie scene"
    ],
    
    # Reflective / moral ending (story with lesson)
    "lesson_ending": [
        "I learned that", "from that day", "never again will I",
        "that taught me", "the moral is", "I now know that",
        "if there's one thing I learned"
    ]
}

In [ ]:
# STEP 63 — DETECT STORYTELLING STYLE
# By counting the presence of storytelling keywords in reviews, we can identify users who tend to write narrative-style reviews, which may correlate with certain personas or sentiments.
def storytelling_score(text):

    text = text.lower()

    score = 0

    for keyword in storytelling_keywords:

        score += text.count(keyword)

    return score

In [ ]:
curated_reviews["storytelling_score"] = (
    curated_reviews["text"]
    .apply(storytelling_score)
)

In [ ]:
# STEP 64 - SLANG & CASUAL LANGUAGE DETECTION

slang_words = [
    
]

slang_words = {

    "frequent": [
        "lol", "omg", "wtf", "super", "kinda", "totally", "literally", "crazy", "weird",
        "awesome", "it's giving"
    ],

    # Pidgin English staples (high‑frequency casual markers)
    "pidgin_staples": [
        "abeg", "na wa o", "wahala", "sef", "nko", "abi", "na", "o",
        "ooh", "sha", "kpa", "kwa", "nawa", "mtchew", "chei", "chai",
        "walahi", "biko", "ndo", "jare", "gan", "sabi"
    ],
    
    # Casual greetings & exclamations
    "exclamations": [
        "see finish", "see me see trouble", "oga", "madam", "boss",
        "my brother", "my sister", "my guy", "my dear", "babe",
        "sis", "bro", "chief", "alhaji", "mama", "papa"
    ],
    
    # Slang for good / bad (informal evaluation)
    "informal_quality": [
        "sweet", "smooth", "soft", "hard", "rough", "wahala",
        "bomb", "lit", "trash", "junk", "wayo", "419", "fake life",
        "original", "oloje", "gbege", "yanfu", "gbese", "sapa"
    ],
    
    # Everyday actions (casual verb forms)
    "casual_verbs": [
        "grab", "chop", "flash", "shayo", "logde", "comot",
        "waka", "manage", "hustle", "hammer", "ball", "settle",
        "run", "form", "fake", "throway", "carry go"
    ],
    
    # Fillers & discourse markers (spoken language features)
    "fillers": [
        "like", "you know", "I mean", "actually", "basically",
        "honestly", "literally", "so yeah", "anyway", "well",
        "the thing is", "see e be like", "make I talk true"
    ],
    
    # Abbreviations & shortenings (text‑speak)
    "abbreviations": [
        "pls", "cos", "bcuz", "u", "ur", "d", "dey", "dem",
        "wen", "den", "nao", "da", "dis", "dat", "dese", "dose",
        "wifi", "data", "app", "phone", "lappy", "btw", "idk", "imo"
    ],
    
    # Repetition for emphasis (casual intensification)
    "repetition": [
        "very very", "too too", "so so", "like like", "many many",
        "plenty plenty", "small small", "quick quick", "everywhere everywhere"
    ],
    
    # Sentence‑final particles (casual tone)
    "final_particles": [
        "o", "ooh", "sha", "na", "abi", "right?", "you get?",
        "you feel me?", "you see?", "ehn?", "not?", "so?"
    ],
    
    # Colloquial time references
    "casual_time": [
        "this morning", "yesterday night", "tomorrow morning",
        "next tomorrow", "today today", "immediately", "straight away",
        "quickly quick", "slowly slowly", "by force by force"
    ],
    
    # Money & transaction slang (very common in reviews)
    "money_slang": [
        "cash", "kudi", "money", "naira", "kobo", "bills", "change",
        "balance", "credit", "loan", "subscription", "airtime", "data"
    ]
}

In [ ]:
# STEP 65 — COMPUTE SLANG SCORE

def slang_score(text):

    text = text.lower()

    score = 0

    for word in slang_words:

        score += text.count(word)

    return score

In [ ]:
curated_reviews["slang_score"] = (
    curated_reviews["text"]
    .apply(slang_score)
)

In [ ]:
# STEP 66 — AGGREGATE LINGUISTIC FEATURES PER USER
# Now we create linguistic identities.
# 

linguistic_features = (
    curated_reviews
    .groupby("user_id")
    [
        [
            "review_length",
            "exclamation_count",
            "question_count",
            "uppercase_ratio",
            "storytelling_score",
            "slang_score"
        ]
    ]
    .mean()
    .reset_index()
)

linguistic_features.head(8)

In [ ]:
# STEP 67 — MERGE INTO PERSONAS

persona_df = persona_df.merge(
    linguistic_features,
    on="user_id",
    how="left"
)

persona_df.head()

In [ ]:
# STEP 68 — CREATE COMMUNICATION STYLES
# Now we interpret linguistic behavior.

def communication_style(row):

    if row["storytelling_score"] > 2:
        return "narrative"

    elif row["review_length"] > 1200:
        return "deeply_descriptive"

    elif row["slang_score"] > 1:
        return "casual_expressive"

    elif row["uppercase_ratio"] > 0.05:
        return "emotionally_emphatic"

    else:
        return "balanced"

In [ ]:
 
persona_df["communication_style"] = (
    persona_df.apply(
        communication_style,
        axis=1
    )
)

persona_df[
    [
        "user_id",
        "archetype",
        "communication_style"
    ]
].head(20)

personas now contain:

- Dimension	(Meaning)
- archetype	(psychological identity)
- dominant_value	(motivations)
- drift	(evolution)
- communication_style	(speaking behavior)

In [ ]:
# STEP 69: VIEW ALL AVAILABLE COMMUNICATION STYLES

persona_df["communication_style"].value_counts()

In [ ]:
# STEP 70: GENERATE USERS BY STYLE
# Narrative Users
narrative_users = persona_df[
    persona_df["communication_style"] == "narrative"
]

narrative_users.head()

In [ ]:
# Casual Expressive Users
casual_users = persona_df[
    persona_df["communication_style"] == "casual_expressive"
]

casual_users.head()

In [ ]:
# Emotionally Emphatic Users
emphatic_users = persona_df[
    persona_df["communication_style"] == "emotionally_emphatic"
]

emphatic_users.head()

In [ ]:
# Deeply Descriptive Users
descriptive_users = persona_df[
    persona_df["communication_style"] == "deeply_descriptive"
]

descriptive_users.head()

In [ ]:
# Balanced Users
balanced_users = persona_df[
    persona_df["communication_style"] == "balanced"
]

balanced_users.head()

In [ ]:
# STEP 71 - View Reviews for a Specific User.. by user id

user_id = "-EX1hrPRBqNkVavtMllTCA"

user_reviews = curated_reviews[
    curated_reviews["user_id"] == user_id
].sort_values("date")

user_reviews[["date", "business_id", "stars", "text"]]

In [ ]:
# Print the reviews nicely

for _, row in user_reviews.iterrows():
    print(row["date"], row["stars"], row["business_id"])
    print(row["text"][:400])
    print("-" * 80)

PHASE 4 is now complete with Linguistic Style Modeling. The personas now understand:

- storytelling behavior, emotional expressiveness, casual language, descriptive depth, communication styles

PHASE 5 — NIGERIAN CONTEXTUALIZATION

BUILD:
1. Create Nigerian linguistic markers
2. Detect local conversational style
3. Create cultural preference signals
4. Build Nigerian behavioral identities
5. Attach localized speaking styles to personas

In [ ]:
# STEP 72 — CREATE NIGERIAN EXPRESSION TAXONOMY
# This will allow us to capture cultural nuances in language use, which can provide deeper insights into user identities, 
# preferences, and sentiments in the context of Nigerian culture.

nigerian_expressions = {

    "soft_life": [
        "soft life", "premium enjoyment", "luxury vibes", "chill spot"
    ],

    "casual_slang": [
        "sha", "abi", "wahala", "dey", "no too bad", "pepper", "gist", "vibes", "omo", "no vex", "no gree for anybody"
    ],

    "pidgin_staples": [
        "abeg", "na wa o", "wahala", "sef", "nko", "abi", "na", "o",
        "ooh", "sha", "kpa", "kwa", "nawa", "mtchew", "chei", "chai",
        "walahi", "biko", "ndo", "jare", "gan", "sabi", "e choke"
    ],
    
    # Casual greetings & exclamations
    "exclamations": [
        "see finish", "see me see trouble", "oga", "madam", "boss",
        "my brother", "my sister", "my guy", "my dear", "babe",
        "sis", "bro", "chief", "alhaji", "mama", "papa"
    ],

    "social_enjoyment": [
        "hangout", "owambe", "groove", "turn up", "outing", "enjoyment" "it's giving", "vibes", "pepper", "gist", "chill spot"
    ],

    # Expressiveness & communication style (Pidgin, humour, directness)
    "expressiveness": [
        "abeg", "na wa o", "seems", "sef", "nko", "abi", "o", "ooh", "omo",
        "who send you", "na so so", "i no send your papa", "e enter"
        "walahi", "mtchew", "chai", "God willing", "not to praise am too much", "you dey whyne?"
    ],

    # Proverbs & wise sayings (often used to justify an opinion)
    "proverbs": [
        "a child who washes hands can eat with elders",
        "the lizard that jumps from a high tree would break its back",
        "when the music changes, the dance must change",
        "the one who throws a stone in the market forgets that others can throw too",
        "a bird that flies off the earth and lands on a tree is not safe from a stone",
        "he who brings kola brings life",
        "the way you dress is how you will be addressed",
        "it is not the size of the yam that matters, but the size of the stew",
        "a person who is chasing a rat cannot see the antelope",
        "if you want to hide a corpse, put it under a woman's wrapper"
    ],
    
    # Idiomatic expressions (figurative, not literal)
    "idioms": [
        "carry last",        # finish last / be embarrassed
        "chop breakfast",    # suffer a harsh disappointment
        "see finish",        # see someone's true colours / be fed up
        "form 419",          # act fraudulent or fake
        "blow grammar",      # speak overly fancy English
        "show pepper",       # be aggressive or tough
        "catch cruise",      # have fun / joke around
        "give attitude",     # behave rudely or arrogantly
        "carry go",          # take away / steal
        "use your head",     # think properly
        "shine your eye",    # be vigilant, don’t be fooled
        "do the needful",    # take necessary action
        "pull down",         # criticise or undermine someone
        "call somebody",     # confront or challenge
        "run mad",           # malfunction / go crazy
        "enter one chance",  # fall into a trap or irreversible situation
        "hot cake",          # very popular in demand
        "no gree for anybody" # stand your ground, don’t give in
    ],
    
    # Greetings & social expressions (used to open or close reviews)
    "greetings": [
        "how far?", "how now?", "how body?", "how market?",
        "hello o", "good morning o", "good afternoon o", "good evening o",
        "thank you jare", "thanks a lot", "appreciate",
        "sorry o", "my bad", "no wahala"
    ],
    
    # Exclamations & emotional outbursts (strong feelings)
    "exclamations": [
        "chai!", "chei!", "mtchew!", "no way!", "kpa!", "nawa o!",
        "God forbid!", "never!", "ehn?", "bawo?", "see glass!", "alas!!",
        "oga at the top!", "hallelujah!", "e shock you?", "e don happen!"
    ],
    
    # Figurative descriptions (vivid, often exaggerated)
    "figurative_descriptions": [
        "hot like suya",           # very hot
        "sweet like honey",        # delicious
        "bitter like agbo",        # very bitter
        "hard like rock",          # extremely hard/tough
        "soft like cotton",        # very soft
        "smooth like butter",      # very smooth
        "long like express",       # very long
        "slow like snail",         # extremely slow
        "fast like wind",          # very fast
        "small like ant",          # tiny
        "full like church on Sunday"  # very crowded
        "you dey whyne?" #are you joking?
    ],
    
    # Conditional & hypothetical phrases (storytelling markers)
    "conditional_phrases": [
        "if to say", "suppose say", "even if", "whether",
        "unless e be say", "as if", "imagine say", "make e be like say"
    ],
    
    # Persuasion & emphasis (used to convince reader)
    "persuasion": [
        "I swear down", "I swear for you", "believe me",
        "take it from me", "mark my word", "I guarantee you",
        "e no go better for you if you doubt", "try me"
    ],
    
    # Blame & criticism expressions
    "blame_criticism": [
        "the fault na", "na him cause am", "who send you?",
        "you no try", "e no correct", "wrong delivery",
        "na scam", "wayo", "419", "fake", "junk", "trash"
    ],
    
    # Humour & sarcasm markers
    "humour_sarcasm": [
        "I laff", "lolz", "mtchew", "see comedy", "joke of the year",
        "e be like film trick", "movie scene", "story for the gods",
        "you won't believe", "as if I never see", "everywhere first blur"
    ],
    
    # Cultural references (places, brands, events)
    "cultural_references": [
        "computer village",           # tech hub in Lagos
        "Alaba market",               # electronics market
        "Oshodi market",              # busy market
        "Balogun market",             # textile market
        "NEPA", "PHCN",               # electricity company
        "MTN", "Glo", "Airtel",       # network providers
        "BBNaija",                    # reality TV show
        "Nollywood",                  # film industry
        "Asake", "Burna Boy",         # popular musicians
        "Dangote",                    # conglomerate
        "Sallah", "Christmas",        # festivals
        "Ember months"                # September–December
    ],

    "practical_survival": [
        "traffic", "expensive", "affordable", "stress", "queue", "delay"
    ],

    "time_efficiency": [
        "wait time", "delay", "fast delivery", "slow", "African time",
        "hours", "minutes", "late", "early", "prompt", "wasted my time",
        "traffic", "Lagos traffic", "delivered on time"
    ],
    
    # Nigerian‑specific storytelling markers (colloquial narrative style)
    "naija_narrative": [
        "so I tell am", "the guy come say", "I just dey go",
        "immediately I enter", "see as e be", "before I know",
        "the next thing", "as I was coming", "I reach there",
        "the seller tell me", "my sister say make I try am",
        "story for another day", "I no fit shout", "you won't believe"
    ],
    
    # Exaggerated storytelling (typical of oral tradition)
    "hyperbole_narrative": [
        "I waited for years", "the longest hour of my life",
        "everybody in Lagos", "the whole market", "I almost died",
        "I swear down", "e be like film", "like a movie scene"
    ],


    # Kinship & relational terms (used to address or refer)
    "kinship": [
        "oga", "madam", "massa", "boss", "chief", "alhaji",
        "my brother", "my sister", "my guy", "my dear", "my friend",
        "uncle", "aunty", "papa", "mama", "baba", "iya", "bros"
    ],
    
    # Conjunctions & connectors (oral style flow)
    "oral_connectors": [
        "so I tell am", "immediately", "the next thing", "before I know",
        "as I dey go", "come see", "lo and behold", "to cut the long story short",
        "long story short", "in short", "and all that", "and so on"
    ]
}

In [ ]:
# STEP 73 — CREATE NIGERIAN STYLE DETECTOR
# By counting the presence of Nigerian-specific expressions in reviews, we can identify users who tend to write in a Nigerian style, which may correlate with certain personas or sentiments.

def detect_nigerian_style(text, taxonomy):

    text = text.lower()

    scores = {}

    for category, phrases in taxonomy.items():

        score = 0

        for phrase in phrases:

            score += text.count(phrase)

        scores[category] = score

    return scores

In [ ]:
# STEP 74 — APPLY NIGERIAN STYLE DETECTION
# This will allow us to identify the presence of Nigerian linguistic and storytelling elements in reviews, providing insights into cultural expression and identity among users.

curated_reviews["nigerian_style"] = (
    curated_reviews["text"]
    .apply(
        lambda x: detect_nigerian_style(
            x,
            nigerian_expressions
        )
    )
)

In [ ]:
# STEP 75 — EXPAND NIGERIAN FEATURES

nigerian_style_df = pd.json_normalize(
    curated_reviews["nigerian_style"]
)

nigerian_style_df.head()

In [ ]:
# STEP 76 — MERGE NIGERIAN FEATURES

curated_reviews = pd.concat(
    [curated_reviews, nigerian_style_df],
    axis=1
)

curated_reviews.head()

In [ ]:
# STEP 77 — AGGREGATE CULTURAL FEATURES PER USER
# This will allow us to aggregate the Nigerian linguistic and storytelling elements at the user level, 
# providing insights into individual cultural expression and identity.

nigerian_features = (
    curated_reviews
    .groupby("user_id")
    [
        list(nigerian_expressions.keys())
    ]
    .mean()
    .reset_index()
)

nigerian_features.head()

In [ ]:
# STEP 78 — MERGE INTO PERSONAS


persona_df = persona_df.merge(
    nigerian_features,
    on="user_id",
    how="left"
)

persona_df.head()

In [ ]:
# STEP 79 — CREATE NIGERIAN CULTURAL IDENTITIES

def nigerian_identity(row):

    if row["soft_life"] > 0.5:
        return "soft_life_explorer"

    elif row["casual_slang"] > 0.5:
        return "casual_slang"
    
    elif row["pidgin_staples"] > 0.5:
        return "pidgin_staples"
    
    elif row["exclamations"] > 0.5:
        return "exclamations"

    elif row["social_enjoyment"] > 0.5:
        return "social_enjoyment"
    
    elif row["expressiveness"] > 0.5:
        return "expressiveness"

    elif row["proverbs"] > 0.5:
        return "proverbs"

    elif row["greetings"] > 0.5:
        return "greetings"

    elif row["exclamations"] > 0.5:
        return "exclamations"

    elif row["figurative_descriptions"] > 0.5:
        return "figurative_descriptions"

    elif row["conditional_phrases"] > 0.5:
        return "conditional_phrases"

    elif row["persuasion"] > 0.5:
        return "persuasion"

    elif row["blame_criticism"] > 0.5:
        return "blame_criticism"

    elif row["humour_sarcasm"] > 0.5:
        return "humour_sarcasm"
    
    elif row["time_efficiency"] > 0.5:
        return "time_efficiency"

    elif row["naija_narrative"] > 0.5:
        return "naija_narrative"

    elif row["hyperbole_narrative"] > 0.5:
        return "hyperbole_narrative"

    elif row["oral_connectors"] > 0.5:
        return "oral_connectors"

    elif row["kinship"] > 0.5:
        return "kinship"
   
    elif row["practical_survival"] > 0.5:
        return "practical_survivor"


    else:
        return "globally_neutral"

In [ ]:
persona_df["nigerian_identity"] = (
    persona_df.apply(
        nigerian_identity,
        axis=1
    )
)

persona_df[
    [
        "user_id",
        "archetype",
        "nigerian_identity"
    ]
].head(10)

FINAL PHASE OF LAYER 1
PHASE 6 — PERSONA SUMMARIZATION & COGNITIVE PROFILES

BUILD:
- Create persona narratives.
- Synthesize traits.
- Create recommendation tendencies.
- Create human-readable summaries.
- Build cognitive identity profiles.

In [ ]:
# STEP 80 — CREATE PERSONA SUMMARY FUNCTION
# This will allow us to generate a concise summary of each user persona.
# By summarizing the key characteristics of each persona, including their archetype, 
# communication style, dominant values, emotional trajectory, and Nigerian identity, 
# we can create a more holistic and human-readable profile for each user.

def generate_persona_summary(row):

    summary = f"""
    Archetype: {row['archetype']}

    Communication Style:
    {row['communication_style']}

    Dominant Value:
    {row['dominant_value']}

    Emotional Trajectory:
    {row['emotional_trajectory']}

    Nigerian Identity:
    {row['nigerian_identity']}
    """

    return summary.strip()

In [ ]:
# STEP 81 — GENERATE PERSONA SUMMARIES
# This will create a human-readable summary for each user persona, encapsulating their key characteristics and insights in a concise format.


persona_df["persona_summary"] = (
    persona_df.apply(
        generate_persona_summary,
        axis=1
    )
)

persona_df[
    [
        "user_id",
        "persona_summary"
    ]
].head()

In [ ]:
# STEP 82 — CREATE RECOMMENDATION TENDENCY ENGINE
# Now we infer what these humans are likely to prefer.

def recommendation_tendency(row):

    dominant_value = row["dominant_value"]
    archetype = row["archetype"]

    if dominant_value == "ambience":
        return "prefers aesthetic and socially vibrant venues"

    elif dominant_value == "service":
        return "prefers highly reliable and respectful experiences"

    elif dominant_value == "food_quality":
        return "prioritizes authentic and high-quality meals"

    elif dominant_value == "affordability":
        return "seeks budget-friendly and high-value experiences"

    elif dominant_value == "social_experience":
        return "prefers lively social environments"

    elif dominant_value == "time_efficiency":
        return "values prompt service and hates unnecessary delays"

    elif dominant_value == "social_proof":
        return "trusts what friends, family, and popular opinion recommend"

    elif dominant_value == "service_quality":
        return "expects courteous, attentive, and reliable staff"

    elif dominant_value == "durability":
        return "prioritises long-lasting, rugged, and original products"

    elif dominant_value == "convenience":
        return "prefers easy access, fast delivery, and hassle‑free processes"

    elif dominant_value == "luxury":
        return "seeks premium, high‑status, and indulgent experiences"
    
    else:
        return "balanced preferences"

In [ ]:
# STEP 83 — APPLY RECOMMENDATION TENDENCIES

persona_df["recommendation_tendency"] = (
    persona_df.apply(
        recommendation_tendency,
        axis=1
    )
)

persona_df[
    [
        "user_id",
        "recommendation_tendency"
    ]
].head()

In [ ]:
# STEP 84 — CREATE FULL COGNITIVE PROFILE
# Now we synthesize the complete behavioral human.

def build_cognitive_profile(row):

    profile = {

        "archetype": row["archetype"],

        "communication_style":
            row["communication_style"],

        "dominant_value":
            row["dominant_value"],

        "emotional_trajectory":
            row["emotional_trajectory"],

        "nigerian_identity":
            row["nigerian_identity"],

        "recommendation_behavior":
            row["recommendation_tendency"]
    }

    return profile

In [ ]:
persona_df["cognitive_profile"] = (
    persona_df.apply(
        build_cognitive_profile,
        axis=1
    )
)

persona_df[
    [
        "user_id",
        "cognitive_profile"
    ]
].head(15)

In [ ]:
# STEP 85 — MANUAL INSPECTION
# Inspect several cognitive profiles.

persona_df[
    [
        "user_id",
        "cognitive_profile"
    ]
].sample(5)

In [ ]:
# View one full cognitive profile

persona_df.iloc[17]["cognitive_profile"]

In [ ]:
# OR

import pprint

pp = pprint.PrettyPrinter(indent=4)

pp.pprint(
    persona_df.iloc[17]["cognitive_profile"]
)

In [ ]:
# OR
 
cognitive_expanded = pd.json_normalize(
    persona_df["cognitive_profile"]
)

cognitive_expanded.head()

In [ ]:
# Pick a user

user_id = persona_df.iloc[4]["user_id"]

# Show profile
pp.pprint(
    persona_df.iloc[0]["cognitive_profile"]
)

# Get reviews
user_reviews = curated_reviews[
    curated_reviews["user_id"] == user_id
]

# Display reviews
user_reviews[
    ["stars", "text"]
].head(3)

Layer 1 is essentially complete. The persona engine and system now understands:

- personality archetypes
- values
- emotional behavior
- temporal evolution
- changing tastes
- communication style
- Nigerian contextual behavior
- recommendation tendencies

That is HUMAN UNDERSTANDING.

In [ ]:
# SAVING PERSONA PROFILES

persona_df.to_csv(
    "../data/processed/persona_profiles.csv",
    index=False
)

In [ ]:
persona_df.head()

In [ ]:
# SAVING CURATED REVIEWS WITH ENRICHED FEATURES

curated_reviews.to_csv(
    "../data/processed/curated_reviews.csv",
    index=False
)

In [ ]:
# SAVING BUSINESS

business_df.to_csv(
    "../data/processed/businesses.csv",
    index=False
)